In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from src.monitoring.logger import load_query_log, setup_logging

setup_logging(log_dir='../logs', console=False)

records = load_query_log('../logs/query_log.jsonl')
queries   = [r for r in records if r.get('type') == 'query']
feedbacks = [r for r in records if r.get('type') == 'feedback']
print(f'Loaded {len(queries)} query records and {len(feedbacks)} feedback records')

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'scripts/analyze_logs.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
latencies = [r['latency_ms'] for r in queries if 'latency_ms' in r]

if latencies:
    plt.figure(figsize=(10, 4))
    plt.hist(latencies, bins=20, color='steelblue', edgecolor='white')
    plt.axvline(sorted(latencies)[int(len(latencies)*0.95)],
                color='red', linestyle='--', label='p95')
    plt.axvline(sum(latencies)/len(latencies),
                color='orange', linestyle='--', label='mean')
    plt.xlabel('Latency (ms)')
    plt.ylabel('Count')
    plt.title('Query latency distribution')
    plt.legend()
    plt.tight_layout()
    plt.savefig('../docs/figures/latency_distribution.png', dpi=150)
    plt.show()
    print(f'p95 latency: {sorted(latencies)[int(len(latencies)*0.95)]:.0f} ms')
else:
    print('No latency data — run some queries first via the Streamlit app or pipeline notebook.')

In [ ]:
if queries:
    df = pd.DataFrame(queries)
    stage_cols = [c for c in ['retrieval_ms','rerank_ms','generation_ms'] if c in df.columns]
    means = df[stage_cols].mean().round(1)
    print('Mean latency by stage:')
    for col, val in means.items():
        print(f'  {col:<16}: {val} ms')

In [ ]:
if queries:
    costs = [r['token_usage'].get('estimated_cost_usd', 0) for r in queries if 'token_usage' in r]
    cumulative = [sum(costs[:i+1]) for i in range(len(costs))]

    plt.figure(figsize=(10, 3))
    plt.plot(cumulative, color='green')
    plt.xlabel('Query number')
    plt.ylabel('Cumulative cost (USD)')
    plt.title(f'Total spend: ${sum(costs):.6f}')
    plt.tight_layout()
    plt.show()

In [ ]:
if queries:
    df = pd.DataFrame(queries)
    print(df['provider'].value_counts().to_string())

In [ ]:
# Run this cell if you have very few log entries to get meaningful charts
from src.pipeline import SRRagPipeline

pipeline = SRRagPipeline(
    persist_dir='../vector_store',
    collection_name='sr_papers',
    llm_provider='mock',
)

test_questions = [
    "What loss function does SRGAN use?",
    "How does SwinIR use transformer attention?",
    "Which papers report results on DIV2K?",
    "What is residual channel attention in RCAN?",
    "Compare SRGAN and ESRGAN architectures.",
    "What makes RealESRGAN handle real-world images?",
    "What PSNR does EDSR achieve on Set5?",
    "How does RDN use dense connections?",
]

print('Running queries to populate log...')
for q in test_questions:
    result = pipeline.query(q, top_k=5)
    print(f'  done: {q[:50]}')

print(f'\nRe-run cells 1-6 to see updated charts.')